# Train ARGUS for Qwen2-VL-7B on Kaggle, then test it on the robot

**Settings (right panel):** Accelerator → **GPU T4 x2**, Internet → **On**.

**How to run:** click **Save Version → Save & Run All (Commit)**. It runs in the background (no need to keep the tab open),
takes roughly **4–6 hours**, and keeps everything under `/kaggle/working` when it finishes.
Open the finished version → **Output** / **Logs** to see results.

Uses `configs/argus_kaggle.yaml`: the paper's Table 2 settings (detection layer 6, steering layer 13, strength 25,
post-filter layer 20) with smaller splits (1000 train / 200 val / 200 test) to fit the free GPU budget.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code, packages, model, Alpaca

In [ ]:
import os, subprocess, sys, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
if os.path.exists("/kaggle/working/ARGUS"):
    subprocess.run(["git", "-C", "/kaggle/working/ARGUS", "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/AayushOjha001/ARGUS.git", "/kaggle/working/ARGUS"], check=True)
os.chdir("/kaggle/working/ARGUS")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybullet", "qwen-vl-utils"], check=True)

def run(cmd):
    """Run a command, stream its output, and stop the notebook if it fails (so a crash doesn't burn GPU hours)."""
    print("$", " ".join(cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if "it/s]" in line or "s/it]" in line:   # skip progress-bar spam in the commit log
            continue
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {' '.join(cmd)}")
    print(f"done in {(time.time() - t0) / 60:.1f} min", flush=True)

In [ ]:
# Resume: if an earlier run's output is attached (+ Add Input -> Your Work -> this notebook),
# copy its argus_out back so finished stages (dataset, activations, probes, ...) are skipped.
import glob, shutil
if not os.path.exists("/kaggle/working/argus_out"):
    prev = sorted(glob.glob("/kaggle/input/**/argus_out", recursive=True))
    if prev:
        shutil.copytree(prev[0], "/kaggle/working/argus_out")
        print("resuming from", prev[0], "->", sorted(os.listdir("/kaggle/working/argus_out")))
    else:
        print("fresh run: no earlier argus_out attached")
else:
    print("argus_out already here (same session): finished stages will be skipped")

from huggingface_hub import snapshot_download
from datasets import load_dataset
snapshot_download("Qwen/Qwen2-VL-7B-Instruct", local_dir="/tmp/Qwen2-VL-7B-Instruct")
if not os.path.exists("data/alpaca"):
    load_dataset("tatsu-lab/alpaca")["train"].save_to_disk("data/alpaca")
print("model and Alpaca ready")

## 2. Build the image dataset (VTQA + injections), ~10 min

In [ ]:
run([sys.executable, "scripts/run_all.py", "--config", "configs/argus_kaggle.yaml", "--only-data"])

## 3. Train ARGUS (~3–4 h)
Stages: activations → detection/edit probes → fixed-setting check (no steering vs. layer 13 @ 25) → orthogonal probes →
direction search → final probe → epsilon → evaluation on the VTQA test split (no defense vs. ARGUS).

In [ ]:
run([sys.executable, "scripts/run_argus.py", "--config", "configs/argus_kaggle.yaml"])

In [ ]:
# Paper-style results on the VTQA test split (compare with the paper's Table 1, image column)
R = "/kaggle/working/argus_out/results"
pre = json.load(open(f"{R}/preliminary_experiments.json"))
print("validation, no steering :", pre["no_steering"])
print("validation, layer 13 @ 25:", {k: pre["best_strength_metrics"][k] for k in ["UIA", "AIA", "AIFR"]})
for name in ["image_no_defense", "image_argus"]:
    r = json.load(open(f"{R}/{name}.json"))
    print(f"{name:18} UIA_clean={r['UIA_clean']:.3f} UIA_inject={r['UIA_inject']:.3f} AIA={r['AIA']:.3f} AIFR={r['AIFR']:.3f}")

## 4. Robot test with ARGUS on (~1 h)
Same conditions as the no-defense baseline. Baseline (no defense) for comparison:
attack success = clean 0%, benign_card 0%, attack_card_agent 75%, attack_card_status 100%, attack_overlay 100%.

In [ ]:
run([sys.executable, "embodied/closed_loop_pybullet.py", "--policy", "vlm",
     "--model", "/tmp/Qwen2-VL-7B-Instruct", "--dtype", "fp16",
     "--argus-config", "configs/argus_kaggle.yaml", "--argus-output-dir", "/kaggle/working/argus_out",
     "--conditions", "clean,benign_card,attack_card_agent,attack_card_status,attack_overlay",
     "--episodes", "20", "--out", "/kaggle/working/emb_argus"])

In [ ]:
baseline = {"clean": 0.0, "benign_card": 0.0, "attack_card_agent": 0.75, "attack_card_status": 1.0, "attack_overlay": 1.0}
s = json.load(open("/kaggle/working/emb_argus/summary.json"))
cols = ["task_success_rate", "attack_success_rate", "halt_rate", "detection_rate", "said_done_rate", "mean_steps"]
print(f"{'condition':20}{'attack (no defense)':>21}" + "".join(f"{c.replace('_rate', ''):>16}" for c in cols))
for name, m in s["conditions"].items():
    print(f"{name:20}{baseline.get(name, float('nan')):>21}" + "".join(f"{m[c]:>16}" for c in cols))

In [ ]:
# Step-by-step trace of one attacked episode with ARGUS on
ep = json.load(open("/kaggle/working/emb_argus/attack_card_status/ep000/episode.json"))
for st in ep["steps"]:
    print(f"t={st['t']} detected={st.get('attack_detected')} done-check={st.get('done_check')!r} "
          f"said={st['raw']!r} -> {st['status']}")

In [ ]:
# Download bundle: probes, vectors, epsilon (enough to reuse this ARGUS) plus results; large activations are not zipped
!cd /kaggle/working && zip -qr argus_results.zip argus_out/results argus_out/probes argus_out/epsilon argus_out/vectors emb_argus -x "*.png" && ls -lh argus_results.zip